# Cell-MSCA Kaggle validation runner

This notebook is a thin orchestration wrapper. Model, split, preprocessing, training, checkpoint, inverse selection, and metrics remain in `src/cell_msca`. It runs generated synthetic data only by default. `/kaggle/input` is treated as read-only and all generated artifacts go under `/kaggle/working`. The test gate remains closed.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)
print('Attached input paths (read-only):')
if KAGGLE_INPUT.is_dir():
    for path in sorted(KAGGLE_INPUT.iterdir()):
        print(' -', path)
        if path.is_dir():
            for child in sorted(path.iterdir())[:20]:
                print('   -', child)
else:
    print(' - /kaggle/input is unavailable; this cell is intended for Kaggle.')


## Select an exact source state

Set `CELL_MSCA_GIT_SHA` to the reviewed commit. Use `CELL_MSCA_SOURCE_MODE=git` when public Git access works, or `attached` for a private Kaggle code dataset. An attached dataset without `.git` must include a small `GIT_COMMIT_SHA.txt` generated when that dataset is packaged. No credentials are read or stored here.

In [ ]:
EXPECTED_GIT_SHA = os.environ.get('CELL_MSCA_GIT_SHA', '').strip().lower()
SOURCE_MODE = os.environ.get('CELL_MSCA_SOURCE_MODE', 'attached').strip().lower()
if len(EXPECTED_GIT_SHA) not in (40, 64):
    raise RuntimeError('Set CELL_MSCA_GIT_SHA to the exact reviewed commit SHA.')

if SOURCE_MODE == 'git':
    REPOSITORY = KAGGLE_WORKING / 'cell-msca-source'
    if REPOSITORY.exists():
        raise FileExistsError(f'Refusing to replace existing source: {REPOSITORY}')
    subprocess.run([
        'git', 'clone', '--no-checkout',
        'https://github.com/subinidus/cell-msca-odiac-estimation.git',
        str(REPOSITORY),
    ], check=True)
    subprocess.run(['git', 'checkout', '--detach', EXPECTED_GIT_SHA], cwd=REPOSITORY, check=True)
    SOURCE_SHA_FILE = None
elif SOURCE_MODE == 'attached':
    candidates = [
        path for path in KAGGLE_INPUT.glob('*')
        if (path / 'src' / 'cell_msca').is_dir()
    ]
    if len(candidates) != 1:
        raise RuntimeError(f'Expected exactly one attached code dataset; found {candidates}')
    ATTACHED_SOURCE = candidates[0]
    REPOSITORY = KAGGLE_WORKING / 'cell-msca-source'
    if REPOSITORY.exists():
        raise FileExistsError(f'Refusing to replace existing source: {REPOSITORY}')
    shutil.copytree(ATTACHED_SOURCE, REPOSITORY)
    declared = os.environ.get('CELL_MSCA_SOURCE_SHA_FILE', '').strip()
    default_sha_file = REPOSITORY / 'GIT_COMMIT_SHA.txt'
    SOURCE_SHA_FILE = Path(declared) if declared else default_sha_file
    if not (REPOSITORY / '.git').exists() and not SOURCE_SHA_FILE.is_file():
        raise RuntimeError('Attached code without .git requires GIT_COMMIT_SHA.txt.')
else:
    raise ValueError("CELL_MSCA_SOURCE_MODE must be 'git' or 'attached'.")

sys.path.insert(0, str(REPOSITORY / 'src'))
print('Repository:', REPOSITORY)
print('Expected Git SHA:', EXPECTED_GIT_SHA)


## Verify the existing environment

This notebook never installs or upgrades PyTorch. The package runner raises a clear error when Python, PyTorch, or NumPy is incompatible. LightGBM is reported for baseline readiness but is not required for this Cell-MSCA-only smoke.

In [ ]:
from cell_msca.kaggle_runner import collect_environment

environment = collect_environment()
for key in ('python', 'pytorch', 'numpy', 'lightgbm', 'cuda_available', 'cuda_version', 'gpu_name'):
    print(f'{key}: {environment[key]}')
if (REPOSITORY / '.git').exists():
    actual_sha = subprocess.run(
        ['git', 'rev-parse', 'HEAD'], cwd=REPOSITORY, check=True,
        capture_output=True, text=True,
    ).stdout.strip().lower()
    if actual_sha != EXPECTED_GIT_SHA:
        raise RuntimeError(f'Git SHA mismatch: {actual_sha} != {EXPECTED_GIT_SHA}')


## Run repository checks

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest'], cwd=REPOSITORY, check=True)
subprocess.run([
    sys.executable, '-m', 'compileall', '-q',
    str(REPOSITORY / 'src'), str(REPOSITORY / 'tests'),
], check=True)


## Generated-data smoke for all four variants

The CPU run is mandatory. When CUDA is available, the same four frozen configurations are also exercised on CUDA. No `v1_legacy` file or test subset is read.

In [ ]:
VARIANTS = ('token_no_attention', 'forward', 'reverse', 'bidirectional')
CONFIG = REPOSITORY / 'configs' / 'kaggle_synthetic_smoke.json'
OUTPUT_ROOT = KAGGLE_WORKING / 'cell-msca-synthetic-validation'

def run_variant(variant, device):
    command = [
        sys.executable, '-m', 'cell_msca.kaggle_runner',
        '--variant', variant, '--train-seed', '3407',
        '--config', str(CONFIG), '--data-path', 'generated',
        '--output-path', str(OUTPUT_ROOT), '--device', device,
        '--expected-git-sha', EXPECTED_GIT_SHA, '--kaggle',
        '--repository-root', str(REPOSITORY),
    ]
    if SOURCE_SHA_FILE is not None and SOURCE_SHA_FILE.is_file():
        command.extend(['--source-git-sha-file', str(SOURCE_SHA_FILE)])
    subprocess.run(command, cwd=REPOSITORY, check=True)

for name in VARIANTS:
    run_variant(name, 'cpu')
if environment['cuda_available']:
    for name in VARIANTS:
        run_variant(name, 'cuda')
else:
    print('CUDA unavailable; CUDA smoke was correctly skipped.')
